# 01. 오디오 특징량과 경량 베이스라인

음성/환경음 모델 개발의 시작점은 파형을 바로 딥러닝에 넣는 것이 아니라, 신호가 어떤 구조를 갖는지 직접 보는 것입니다.
이 노트북에서는 합성 신호로 RMS, zero-crossing rate, 스펙트럼 중심, 로그 스펙트럼을 만들고 간단한 분류기를 학습합니다.

논문 연결: 최종 논문의 `Baseline`과 `Feature/Preprocessing` 절에 들어갈 최소 비교군을 만듭니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
sr = 16_000
duration = 1.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

def make_signal(kind: str, freq=440, noise=0.02):
    if kind == "tone":
        y = np.sin(2 * np.pi * freq * t)
    elif kind == "chirp":
        y = np.sin(2 * np.pi * (freq + 600 * t) * t)
    elif kind == "pulse":
        y = ((np.sin(2 * np.pi * freq * t) > 0.8).astype(float) * 2) - 1
    elif kind == "noise":
        y = np.random.normal(0, 1, len(t))
    else:
        raise ValueError(kind)
    return y + np.random.normal(0, noise, len(t))

signals = []
for label in ["tone", "chirp", "pulse", "noise"]:
    for i in range(40):
        signals.append((label, make_signal(label, freq=random.choice([220, 440, 880]))))

if plt:
    fig, axes = plt.subplots(2, 2, figsize=(10, 5))
    for ax, (label, y) in zip(axes.ravel(), signals[::40]):
        ax.plot(t[:800], y[:800])
        ax.set_title(label)
    fig.tight_layout()


In [ ]:
def audio_features(y, sr=16_000):
    eps = 1e-9
    rms = float(np.sqrt(np.mean(y ** 2)))
    zcr = float(np.mean(np.abs(np.diff(np.signbit(y)))))
    spectrum = np.abs(np.fft.rfft(y))
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    centroid = float((freqs * spectrum).sum() / (spectrum.sum() + eps))
    bandwidth = float(np.sqrt((((freqs - centroid) ** 2) * spectrum).sum() / (spectrum.sum() + eps)))
    rolloff_idx = int(np.searchsorted(np.cumsum(spectrum), 0.85 * spectrum.sum()))
    rolloff = float(freqs[min(rolloff_idx, len(freqs) - 1)])
    return [rms, zcr, centroid, bandwidth, rolloff]

rows = []
for label, y in signals:
    rows.append([label] + audio_features(y, sr))
df = pd.DataFrame(rows, columns=["label", "rms", "zcr", "centroid", "bandwidth", "rolloff"])
df.to_csv(ARTIFACTS / "audio_features.csv", index=False, encoding="utf-8-sig")
df.groupby("label").mean(numeric_only=True)


In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.metrics import classification_report, confusion_matrix

    X = df.drop(columns=["label"])
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    report = classification_report(y_test, pred, output_dict=True)
    save_json("audio_baseline_report.json", report)
    print(classification_report(y_test, pred))
    print(confusion_matrix(y_test, pred, labels=sorted(y.unique())))
except Exception as exc:
    print("scikit-learn이 없으면 requirements-minimal.txt 설치 후 다시 실행하세요:", exc)


## 확장 과제

- 실제 음성 클립에서 위 특징량을 추출하고, 합성 신호와 분포 차이를 비교합니다.
- 논문용 그림으로 `label별 centroid/rolloff boxplot`을 추가합니다.
- 모델이 틀린 샘플의 파형과 스펙트럼을 함께 저장합니다.
